In [2]:
# train_svdpp.py
from surprise import Dataset, Reader, SVDpp
import pickle

# 1) Load your base ratings CSV
reader = Reader(line_format="user item rating timestamp", sep=",", skip_lines=1)
data   = Dataset.load_from_file("ml-latest-small/ratings.csv", reader=reader)

# 2) Build full trainset & fit SVD++
trainset = data.build_full_trainset()
algo = SVDpp(
    n_factors=50,    # number of latent dims
    lr_all=0.005,    # learning rate
    reg_all=0.02,    # regularization
    verbose=True,
    n_jobs=-1,
)
algo.fit(trainset)

# 3) Save the fitted model
with open("models/svdpp.pkl","wb") as f:
    pickle.dump(algo, f)

print("SVD++ trained.")


 processing epoch 0
 processing epoch 1
 processing epoch 2
 processing epoch 3
 processing epoch 4
 processing epoch 5
 processing epoch 6
 processing epoch 7
 processing epoch 8
 processing epoch 9
 processing epoch 10
 processing epoch 11
 processing epoch 12
 processing epoch 13
 processing epoch 14
 processing epoch 15
 processing epoch 16
 processing epoch 17
 processing epoch 18
 processing epoch 19
SVD++ trained.


In [3]:
import pickle
from surprise import SVDpp

# load
with open("models/svdpp.pkl","rb") as f:
    algo = pickle.load(f)

# predict for a brand-new user "alice" on movie "1234"
pred = algo.predict(uid="alice", iid="1234")
print(f"Est. rating = {pred.est:.2f}")


Est. rating = 3.88


In [ ]:
# fold_in.py

import numpy as np
import pandas as pd
import pickle

def fold_in_user(algo, user_ratings, reg=0.1):
    """
    Given a trained SVDpp algo and a dict of {movieId: rating} for a new user,
    return the user's latent-factor vector p_u via ridge regression.
    """
    trainset = algo.trainset
    mu       = trainset.global_mean

    # 1) Build item-bias map: raw_iid (str) -> bias
    bi = {
        trainset.to_raw_iid(inner_i): bias
        for inner_i, bias in enumerate(algo.bi)
    }

    # 2) Collect only movies the user rated AND that the trainset knows
    Q_rows = []
    y = []
    for raw_mid, r_ui in user_ratings.items():
        raw_mid_str = str(raw_mid)
        try:
            inner_i = trainset.to_inner_iid(raw_mid_str)
        except ValueError:
            # this movie wasn't in training, skip it
            continue

        # qi is shape (n_items, n_factors)
        q_i = algo.qi[inner_i]  
        # target is rating minus global + item bias
        target = r_ui - mu - bi[raw_mid_str]

        Q_rows.append(q_i)
        y.append(target)

    if not Q_rows:
        raise ValueError("No overlap between user_ratings and trained items.")

    Q = np.vstack(Q_rows)       # shape (n_rated, n_factors)
    y = np.array(y)             # shape (n_rated,)

    # 3) Solve (QᵀQ + reg·I) p_u = Qᵀ y
    A = Q.T.dot(Q) + reg * np.eye(algo.n_factors)
    b = Q.T.dot(y)
    p_u = np.linalg.solve(A, b)

    return p_u


def recommend_new_user(algo, p_u, movies_csv, top_n=10):
    """
    Given p_u for a new user, score all items and return top_n (title,score).
    """
    trainset = algo.trainset
    mu       = trainset.global_mean

    # Load movieId→title
    movies = pd.read_csv(movies_csv, dtype={"movieId": str})

    # Build a list of (raw_iid, q_i) for all items
    results = []
    for inner_i, q_i in enumerate(algo.qi):
        raw_i = trainset.to_raw_iid(inner_i)
        b_i   = algo.bi[inner_i]
        est   = mu + b_i + q_i.dot(p_u)
        results.append((raw_i, est))

    # Pick top_n
    top = sorted(results, key=lambda x: x[1], reverse=True)[:top_n]

    # Map raw_i → title
    recs = []
    for raw_i, score in top:
        title = movies.loc[movies.movieId == raw_i, "title"].squeeze()
        recs.append((title, round(score, 2)))

    return recs


if __name__ == "__main__":
    # 1) load model
    with open("models/svdpp.pkl", "rb") as f:
        algo = pickle.load(f)

    user_df = pd.read_csv("alex_data/ratings_with_tmdb.csv")  # userId,tmdbId,Rating
    #    merge to get MovieLens movieId
    links   = pd.read_csv("ml-latest-small/links.csv")  # movieId,tmdbId
    merged  = user_df.merge(links, left_on="tmdb_id", right_on="tmdbId")
    user_ratings = dict(zip(merged.movieId, merged.Rating))

    # # 2) new user ratings as { movieId: rating }
    # #    NOTE: keys can be ints or strings; they will be cast to str internally.
    # example_ratings = {1: 4.0, 50: 3.5, 258: 5.0}

    # 3) fold in & recommend
    pu   = fold_in_user(algo, user_ratings, reg=0.1)
    recs = recommend_new_user(
        algo, pu,
        movies_csv="ml-latest-small/movies.csv",
        top_n=10
    )

    recs_no_dupes = recommend_new_user_no_dupes(
        algo, pu,
        movies_csv="ml-latest-small/movies.csv",
        movies_csv="ml-latest-small/watched_with_tmdb.csv",
        top_n=10
    )

    print("Top 10 personalized recs:")
    for title, score in recs:
        print(f" • {title} — {score}★")

    print("Top 10 personalized recs_no_dupes:")
    for title, score in recs_no_dupes:
        print(f" • {title} — {score}★")


Top 10 personalized recs:
 • Pulp Fiction (1994) — 5.38★
 • Shawshank Redemption, The (1994) — 5.27★
 • Lives of Others, The (Das leben der Anderen) (2006) — 5.14★
 • Green Mile, The (1999) — 5.1★
 • Eternal Sunshine of the Spotless Mind (2004) — 5.08★
 • Chinatown (1974) — 4.96★
 • Hours, The (2002) — 4.96★
 • Fisher King, The (1991) — 4.94★
 • City of Lost Children, The (Cité des enfants perdus, La) (1995) — 4.94★
 • Ran (1985) — 4.93★


In [ ]:
import pandas as pd

def recommend_new_user_no_dupes(algo, p_u, movies_csv, links_csv, watched_csv, top_n=10):
    """
    Generate recommendations for a new user, excluding movies they have already watched.
    
    Parameters:
    - algo: Trained Surprise SVD++ algorithm.
    - p_u: Latent factors for the new user.
    - movies_csv: Path to the movies.csv file (contains movieId and title).
    - links_csv: Path to the links.csv file (contains movieId and tmdbId).
    - watched_csv: Path to the watched_with_tmdb.csv file (contains tmdbId of watched movies).
    - top_n: Number of recommendations to return.
    
    Returns:
    - List of top_n recommendations as (title, score) tuples.
    """
    trainset = algo.trainset
    mu = trainset.global_mean

    # Load movieId → title mapping
    movies = pd.read_csv(movies_csv, dtype={"movieId": str})

    # Load watched movies and map tmdbId → movieId
    watched = pd.read_csv(watched_csv, dtype={"tmdb_id": str})
    print(watched)
    print(f"Number of movies in watched before merge: {len(watched)}")
    
    links = pd.read_csv(links_csv, dtype={"tmdbId": str, "movieId": str})
    watched = watched.merge(links, left_on="tmdb_id", right_on="tmdbId", how="inner")
    watched_movie_ids = set(watched["movieId"])
    

    print(f"Number of movies in watched after merge: {len(watched_movie_ids)}")

    print(watched_movie_ids)

    # Build a list of (raw_iid, q_i) for all items
    results = []
    for inner_i, q_i in enumerate(algo.qi):
        raw_i = trainset.to_raw_iid(inner_i)
        if raw_i in watched_movie_ids:
            continue  # Skip movies the user has already watched
        b_i = algo.bi[inner_i]
        est = mu + b_i + q_i.dot(p_u)
        results.append((raw_i, est))

    # Pick top_n recommendations
    top = sorted(results, key=lambda x: x[1], reverse=True)[:top_n]

    # Map raw_i → title
    recs = []
    for raw_i, score in top:
        title = movies.loc[movies.movieId == raw_i, "title"].squeeze()
        recs.append((title, round(score, 2)))

    return recs

# Example usage
if __name__ == "__main__":
    # Load the trained model
    with open("models/svdpp.pkl", "rb") as f:
        algo = pickle.load(f)

    # Generate recommendations
    recs = recommend_new_user_no_dupes(
        algo,
        p_u,
        movies_csv="ml-latest-small/movies.csv",
        links_csv="ml-latest-small/links.csv",
        watched_csv="alex_data/watched_with_tmdb.csv",
        top_n=10,
    )

    print("Top 10 recommendations (excluding watched movies):")
    for title, score in recs:
        print(f" • {title} — {score}★")

           Date                   Name    Year        Letterboxd URI  \
0    2021-07-28                    Pig  2021.0  https://boxd.it/nzoW   
1    2021-07-28     Bo Burnham: Inside  2021.0  https://boxd.it/v2uy   
2    2021-07-28               Parasite  2019.0  https://boxd.it/hTha   
3    2021-07-28              Midsommar  2019.0  https://boxd.it/jhxe   
4    2021-07-28                  Joker  2019.0  https://boxd.it/h4cS   
..          ...                    ...     ...                   ...   
536  2025-01-21    Hundreds of Beavers  2022.0  https://boxd.it/CyUM   
537  2025-01-21  Manchester by the Sea  2016.0  https://boxd.it/b2L0   
538  2025-01-21          Fallen Angels  1995.0  https://boxd.it/1UkW   
539  2025-02-16       One of Them Days  2025.0  https://boxd.it/MReU   
540  2025-03-02           Annihilation  2018.0  https://boxd.it/9yzm   

       tmdb_id  
0     635731.0  
1     823754.0  
2      48311.0  
3     530385.0  
4     475557.0  
..         ...  
536  1019939.0  